# Notebook 04 — Network Priors, Crosstalk, and Entity Masks

Three network matrices encode biological prior knowledge into the ODE:

| Matrix | Shape | Meaning |
|--------|-------|---------|
| `Cg` | N × N | Global PTM crosstalk (from database) |
| `Cl` | N × N | Local sequence-proximity crosstalk |
| `K_site_kin` | N × M | Kinase → phosphosite weight |


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from phoscrosstalk.data_loader import (
    load_site_data, load_kinase_site_matrix, load_tf_network,
    build_tf_prot_weights, row_normalize
)

timepoints = list(range(1, 15))
sites, proteins, site_prot_idx, positions, t, Y, A_data, A_proteins = \
    load_site_data(SAMPLE_DIR / "protephospho.csv", timepoints)
N = len(sites)
print(f"sites: {sites}")
print(f"N = {N} phosphosites, K = {len(proteins)} proteins")


## 1 · Cg and Cl — PTM crosstalk matrices

In [ ]:
# PTM database files (SQLite) are not included in sample_data.
# We build placeholder zeros to demonstrate the expected structure.

Cg = np.zeros((N, N), dtype=float)  # global crosstalk — zeros = no DB info
Cl = np.zeros((N, N), dtype=float)  # local proximity crosstalk — build from positions

# Build Cl from residue positions (exponential decay with sequence distance)
length_scale = 50.0
for i in range(N):
    for j in range(N):
        if i == j:
            continue
        if site_prot_idx[i] != site_prot_idx[j]:
            continue  # only intra-protein
        if np.isfinite(positions[i]) and np.isfinite(positions[j]):
            d = abs(positions[i] - positions[j])
            Cl[i, j] = np.exp(-d / length_scale)

print("Cg shape:", Cg.shape, "(all zeros — no PTM DB)")
print("Cl shape:", Cl.shape)
print("Cl[non-zero entries]:")
for i in range(N):
    for j in range(N):
        if Cl[i, j] > 0:
            print(f"  Cl[{sites[i]}, {sites[j]}] = {Cl[i,j]:.4f}  (|pos diff| = {abs(positions[i]-positions[j]):.0f})")


## 2 · K_site_kin — kinase–site weight matrix

In [ ]:
K_site_kin, kinases = load_kinase_site_matrix(SAMPLE_DIR / "kinase_sites.tsv", sites)
M = len(kinases)

print(f"K_site_kin shape: {K_site_kin.shape}  (N_sites × N_kinases)")
print(f"kinases: {kinases}")
print()
df_kskin = pd.DataFrame(K_site_kin, index=sites, columns=kinases)
print(df_kskin.round(4))


## 3 · Row-normalisation

In [ ]:
K_site_kin_norm = row_normalize(K_site_kin)
print("After row_normalize — each row sums to ≤1:")
df_norm = pd.DataFrame(K_site_kin_norm, index=sites, columns=kinases)
print(df_norm.round(4))
print("\nRow sums:", K_site_kin_norm.sum(axis=1).round(4))


## 4 · R matrix — kinase feedback support

In [ ]:
# R = K_site_kin.T  →  shape (M, N)
# R[m, n] answers: "how strongly does kinase m support site n?"
R = K_site_kin_norm.T
print(f"R shape: {R.shape}  (M_kinases × N_sites)")
df_R = pd.DataFrame(R, index=kinases, columns=sites)
print(df_R.round(4))


## 5 · L_alpha — kinase network Laplacian

In [ ]:
# L_alpha regularises kinase activation via a graph Laplacian.
# If no kinase–kinase graph is available, use a zero matrix (no regularisation).
L_alpha = np.zeros((M, M), dtype=float)
print(f"L_alpha shape: {L_alpha.shape}  (M × M)  — zero = no kinase graph prior")


## 6 · site_prot_idx — entity mask

In [ ]:
print("site_prot_idx maps each phosphosite to its parent protein:")
for s, k in zip(sites, site_prot_idx):
    print(f"  site {s:<18} → protein index {k}  ({proteins[k]})")

# Visualise as a binary membership matrix
membership = np.zeros((N, len(proteins)), dtype=int)
for n, k in enumerate(site_prot_idx):
    membership[n, k] = 1

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(membership, cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(proteins)))
ax.set_xticklabels(proteins, rotation=30, ha="right")
ax.set_yticks(range(N))
ax.set_yticklabels(sites, fontsize=8)
ax.set_title("Site–protein membership (site_prot_idx)")
ax.set_xlabel("protein")
ax.set_ylabel("phosphosite")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_site_protein_membership.png", dpi=100)
plt.show()


## 7 · TF network — tf_prot_weights

In [ ]:
tf_df = load_tf_network(SAMPLE_DIR / "tf_mrna.csv", gene_ids=proteins)
print("TF network dataframe:")
print(tf_df)
print()

tf_weights = build_tf_prot_weights(tf_df, gene_ids=proteins, proteins=proteins)
print("tf_prot_weights shape:", tf_weights.shape, "  (K × K)  — TF → protein synthesis weight")
print(pd.DataFrame(tf_weights, index=proteins, columns=proteins).round(4))


## 8 · Kinase–site biology

In [ ]:
print("Kinase–substrate relationships in sample data:")
for s in sites:
    prot, residue = s.split("_", 1)
    row = K_site_kin_norm[sites.index(s)]
    kinase_str = ", ".join(f"{kinases[j]}({row[j]:.2f})" for j in range(M) if row[j] > 0)
    print(f"  {s:<20} phosphorylated by: {kinase_str or 'none'}")
